# Neural Network + FGM Pipeline

This notebook follows the handwritten pipeline and uses the model settings loaded from `MachineLearning\NeuralNetworks\best_params.csv`.

Pipeline steps:

1. `clean.fit(X_train, y_train)` to train the clean model  
2. `clean.predict(X_test)` and evaluate the clean model on clean test data  
3. Save the clean model  
4. Initialize the FGM attack  
5. Generate adversarial train and test samples  
6. Keep adversarial labels aligned with the original ground-truth labels  
7. Build combined clean+adversarial train and test sets  
8. Retrain using ART's `AdversarialTrainer` with the FGM attack  
9. Evaluate both the clean model and the adversarially trained model on:
   - clean test data
   - adversarial test data
   - combined clean+adversarial test data


In [87]:
# If needed, install once:
# !pip install torch scikit-learn adversarial-robustness-toolbox pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer


In [ ]:
BEST_PARAMS_PATH = Path(r"..\MachineLearning\NeuralNetworks\best_params.csv")

def parse_hidden_layers(value):
    if isinstance(value, (tuple, list)):
        return tuple(int(v) for v in value)

    text = str(value).strip().strip('"').strip("'")
    if text.startswith("(") or text.startswith("["):
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)):
            return tuple(int(v) for v in parsed)

    return tuple(int(part.strip()) for part in text.split(",") if part.strip())

if not BEST_PARAMS_PATH.exists():
    raise ValueError(f"File not found: {BEST_PARAMS_PATH}")

best_params_df = pd.read_csv(BEST_PARAMS_PATH)
BEST_PARAMS = best_params_df.iloc[0].to_dict()
BEST_PARAMS["hidden_layer_sizes"] = parse_hidden_layers(BEST_PARAMS["hidden_layer_sizes"])
BEST_PARAMS["alpha"] = float(BEST_PARAMS["alpha"])
BEST_PARAMS["learning_rate_init"] = float(BEST_PARAMS["learning_rate_init"])
BEST_PARAMS["batch_size"] = int(BEST_PARAMS["batch_size"])
BEST_PARAMS["max_iter"] = int(BEST_PARAMS["max_iter"])
BEST_PARAMS["early_stopping"] = bool(BEST_PARAMS["early_stopping"])
BEST_PARAMS["n_iter_no_change"] = int(BEST_PARAMS["n_iter_no_change"])
BEST_PARAMS["random_state"] = int(BEST_PARAMS["random_state"])

SEED = BEST_PARAMS["random_state"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEFAULT_DATA_PATH = Path(r"..\CSVs\newDataset.csv")
RUNS_DIR = Path(r"..\StandardizedRuns")
RUN_GLOB = "NeuralNet_train_*.csv"

# Optional environment overrides:
# - NN_RUN_PATH
# - MODEL_RUN_PATH
ENV_RUN_PATH = os.environ.get("NN_RUN_PATH") or os.environ.get("MODEL_RUN_PATH")

LABEL_COL = "anomaly"
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.75
BATCH_SIZE = BEST_PARAMS["batch_size"]
NB_EPOCHS = BEST_PARAMS["max_iter"]
LR = BEST_PARAMS["learning_rate_init"]
WEIGHT_DECAY = BEST_PARAMS["alpha"]
HIDDEN_LAYER_SIZES = BEST_PARAMS["hidden_layer_sizes"]
ACTIVATION_NAME = str(BEST_PARAMS["activation"]).lower()
SOLVER_NAME = str(BEST_PARAMS["solver"]).lower()
LR_POLICY = str(BEST_PARAMS["learning_rate"]).lower()
EARLY_STOPPING = BEST_PARAMS["early_stopping"]
N_ITER_NO_CHANGE = BEST_PARAMS["n_iter_no_change"]

FGM_EPS = 0.10

SAVE_MODELS = False
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_path() -> Path:
    if ENV_RUN_PATH:
        candidate = Path(ENV_RUN_PATH)
        if candidate.exists():
            return candidate
        print(f"[warn] Env path not found: {candidate}")

    if RUNS_DIR.exists():
        candidates = sorted(
            RUNS_DIR.glob(RUN_GLOB),
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            return candidates[0]

    return DEFAULT_DATA_PATH

DATA_PATH = resolve_data_path()
print(f"Using data source: {DATA_PATH}")
print("Loaded best params:")
print(BEST_PARAMS)

Using data source: ..\StandardizedRuns\NeuralNet_train_clean.csv
Loaded best params:
{'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}


In [76]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()
    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )
    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler

X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(str(DATA_PATH))

Loaded: ..\StandardizedRuns\NeuralNet_train_clean.csv
Rows=1698, Features=18, Label dist=[1351  347]
Train=(424, 18), Test=(1274, 18)


In [77]:
def get_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU
    if name == "tanh":
        return nn.Tanh
    if name == "logistic":
        return nn.Sigmoid
    raise ValueError(f"Unsupported activation for this notebook: {name}")

class MLP(nn.Module):
    def __init__(self, d_in: int, hidden_layer_sizes=HIDDEN_LAYER_SIZES, activation_name: str = ACTIVATION_NAME):
        super().__init__()

        activation_cls = get_activation(activation_name)
        layers = []
        in_features = d_in

        for hidden_units in hidden_layer_sizes:
            layers.append(nn.Linear(in_features, int(hidden_units)))
            layers.append(activation_cls())
            in_features = int(hidden_units)

        layers.append(nn.Linear(in_features, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_art_classifier(
    d_in: int,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    activation_name: str = ACTIVATION_NAME,
):
    model = MLP(
        d_in=d_in,
        hidden_layer_sizes=hidden_layer_sizes,
        activation_name=activation_name,
    )
    criterion = nn.CrossEntropyLoss()


    return PyTorchClassifier(
        model=model,
        loss=criterion,
        optimizer=optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay),
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


def predict_labels(art_clf: PyTorchClassifier, X: np.ndarray):
    probs = art_clf.predict(X)
    return np.argmax(probs, axis=1)


def eval_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


def eval_classifier(art_clf: PyTorchClassifier, X: np.ndarray, y: np.ndarray, name: str):
    y_pred = predict_labels(art_clf, X)
    metrics = eval_from_predictions(y, y_pred, name)
    return metrics, y_pred


def save_art_model_state(art_clf: PyTorchClassifier, out_path: Path):
    torch.save(art_clf.model.state_dict(), out_path)
    print(f"Saved model state: {out_path}")


In [78]:
# Step 1: clean.fit(X_train, y_train) -> trained clean model
print('Training clean model with:', {
    'hidden_layer_sizes': HIDDEN_LAYER_SIZES,
    'alpha': WEIGHT_DECAY,
    'learning_rate_init': LR,
    'batch_size': BATCH_SIZE,
    'activation': ACTIVATION_NAME,
    'solver': SOLVER_NAME,
    'learning_rate': LR_POLICY,
    'max_iter': NB_EPOCHS,
    'early_stopping': EARLY_STOPPING,
    'n_iter_no_change': N_ITER_NO_CHANGE,
    'random_state': SEED,
})

art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

# y_pred = clean.predict(X_test)
# eval_classifier(clean, X_test, y_pred)
clean_on_clean, y_pred_clean = eval_classifier(
    art_clean,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

# save clean model
if SAVE_MODELS:
    save_art_model_state(art_clean, ARTIFACT_DIR / "nn_clean_model.pt")


Training clean model with: {'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}

[clean_model_on_clean_test] acc=0.9474 f1=0.8607
confusion matrix:
[[1000   14]
 [  53  207]]
              precision    recall  f1-score   support

           0     0.9497    0.9862    0.9676      1014
           1     0.9367    0.7962    0.8607       260

    accuracy                         0.9474      1274
   macro avg     0.9432    0.8912    0.9141      1274
weighted avg     0.9470    0.9474    0.9458      1274



In [79]:
# Step 2: initialize FGM and generate adversarial samples
fgm = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)

X_train_adv = fgm.generate(x=X_train)
X_test_adv = fgm.generate(x=X_test)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# If y_train_adv is needed, predict it.
# The handwritten pipeline notes this as optional.
y_train_adv_pred = predict_labels(art_clean, X_train_adv)
y_test_adv_pred = predict_labels(art_clean, X_test_adv)

# For retraining and evaluation targets, keep the original ground-truth labels.
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

# Build combined clean + adversarial datasets.
X_train_combined = np.concatenate([X_train, X_train_adv], axis=0).astype(np.float32)
y_train_combined = np.concatenate([y_train, y_train_adv], axis=0).astype(np.int64)

X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)

print("Combined datasets created:")
print("X_train_combined:", X_train_combined.shape)
print("y_train_combined:", y_train_combined.shape)
print("X_test_combined:", X_test_combined.shape)
print("y_test_combined:", y_test_combined.shape)


Adversarial data generated:
X_train_adv: (424, 18)
X_test_adv: (1274, 18)
Combined datasets created:
X_train_combined: (848, 18)
y_train_combined: (848,)
X_test_combined: (2548, 18)
y_test_combined: (2548,)


In [80]:
# Evaluate performance of the clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_classifier(
    art_clean,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_classifier(
    art_clean,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.1397 f1=0.1672
confusion matrix:
[[ 68 946]
 [150 110]]
              precision    recall  f1-score   support

           0     0.3119    0.0671    0.1104      1014
           1     0.1042    0.4231    0.1672       260

    accuracy                         0.1397      1274
   macro avg     0.2080    0.2451    0.1388      1274
weighted avg     0.2695    0.1397    0.1220      1274


[clean_model_on_combined_test] acc=0.5436 f1=0.3528
confusion matrix:
[[1068  960]
 [ 203  317]]
              precision    recall  f1-score   support

           0     0.8403    0.5266    0.6475      2028
           1     0.2482    0.6096    0.3528       520

    accuracy                         0.5436      2548
   macro avg     0.5443    0.5681    0.5001      2548
weighted avg     0.7195    0.5436    0.5873      2548



In [81]:
# Retrain using ART's AdversarialTrainer with the FGM attack
art_adv = make_art_classifier(d_in=X_train.shape[1])

adv_trainer = AdversarialTrainer(
    classifier=art_adv,
    attacks=fgm,
    ratio=.55,  # Use 50% adversarial samples during training
)

adv_trainer.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    nb_epochs=NB_EPOCHS,
)

if SAVE_MODELS:
    save_art_model_state(art_adv, ARTIFACT_DIR / "nn_adversarial_trained_model.pt")


Adversarial training epochs: 100%|██████████| 1000/1000 [00:23<00:00, 41.97it/s]


In [82]:
# Test using X_test_adv and the combined clean+adv test set
adv_trained_on_adv, y_pred_adv = eval_classifier(
    art_adv,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

# Also check whether adversarial training preserved clean performance
adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
    art_adv,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
    art_adv,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)



[adv_trained_model_on_adv_test] acc=0.9396 f1=0.8344
confusion matrix:
[[1003   11]
 [  66  194]]
              precision    recall  f1-score   support

           0     0.9383    0.9892    0.9630      1014
           1     0.9463    0.7462    0.8344       260

    accuracy                         0.9396      1274
   macro avg     0.9423    0.8677    0.8987      1274
weighted avg     0.9399    0.9396    0.9368      1274


[adv_trained_model_on_clean_test] acc=0.9466 f1=0.8559
confusion matrix:
[[1004   10]
 [  58  202]]
              precision    recall  f1-score   support

           0     0.9454    0.9901    0.9672      1014
           1     0.9528    0.7769    0.8559       260

    accuracy                         0.9466      1274
   macro avg     0.9491    0.8835    0.9116      1274
weighted avg     0.9469    0.9466    0.9445      1274


[adv_trained_model_on_combined_test] acc=0.9431 f1=0.8453
confusion matrix:
[[2007   21]
 [ 124  396]]
              precision    recall  f1-scor

In [83]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_clean,
    adv_trained_on_adv,
    adv_trained_on_combined,
])

summary_df


,model_eval,acc,f1
0,clean_model_on_clean_test,0.947410,0.860707
1,clean_model_on_adv_test,0.139717,0.167173
2,clean_model_on_combined_test,0.543564,0.352810
3,adv_trained_model_on_clean_test,0.946625,0.855932
4,adv_trained_model_on_adv_test,0.939560,0.834409
5,adv_trained_model_on_combined_test,0.943093,0.845251


In [84]:
# Save metrics
out_csv = Path(r"Results\NeuralNetworksResults\nn_fgm_evasion_pipeline_.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


Saved: Results\NeuralNetworksResults\nn_fgm_evasion_pipeline_.csv


## Dual-Stream Consistency Gate

This section now mirrors the **dual-stream detector flow** used in the Transformer, LSTM, and CNN notebooks.

### What changed
- Treat the clean model as the **Nominal Model**
- Treat the adversarially trained model as the **Guardian Model**
- Detect possible attacks using:
  1. **Prediction disagreement**
  2. **High-confidence disagreement**
  3. A guarded case where the nominal model predicts benign and the guardian predicts anomaly

### Why this is better
The earlier gate converted both model outputs into three custom labels (`Anomaly`, `AttackFlag`, `Nominal`), which did not match the other three pipelines. This version evaluates the gate as an **attack detector**, so you can report:
- **FPR** on clean samples
- **TPR** on adversarial samples
- **F1** for clean-vs-attack detection

It also returns the **final gated prediction**, where the guardian prediction is used when an attack is detected and the nominal prediction is used otherwise.


In [85]:

# Dual-stream consistency gate aligned with the CNN / LSTM / Transformer notebooks

CONFIDENCE_THRESHOLD = 0.20
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(art_clf: PyTorchClassifier, X: np.ndarray):
    probs = art_clf.predict(X).astype(np.float32)
    preds = np.argmax(probs, axis=1)
    return preds, probs

class DualStreamDetector:
    """
    Dual-stream detector for the NN pipeline.

    Components:
    - Nominal model: trained only on clean data
    - Guardian model: adversarially trained model
    - Consistency gate: flags likely attacks based on disagreement
    """

    def __init__(self, nominal_model: PyTorchClassifier, guardian_model: PyTorchClassifier):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(pA[yA])
            conf_guardian = float(pB[yB])

            detected = False
            reasons = []

            # Rule 1: any prediction disagreement is suspicious
            if yA != yB:
                detected = True
                reasons.append("disagreement")

            # Rule 2: nominal says benign, guardian says anomaly, both are confident,
            # and the guardian anomaly probability is sufficiently larger
            if yA == 0 and yB == 1:
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    prob_diff = float(pB[1] - pA[1])
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")
                else:
                    prob_diff = float(pB[1] - pA[1])
            else:
                prob_diff = float(pB[1] - pA[1])

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df

def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }

detector = DualStreamDetector(
    nominal_model=art_clean,
    guardian_model=art_adv,
)

# Evaluate final gated predictions on clean, adversarial, and combined sets
gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("\nDual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

# Evaluate the gate specifically as an attack detector
y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "FGM",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("\nDual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("\nGate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("\nGate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))



[dual_stream_final_predictions_on_clean_test] acc=0.9466 f1=0.8559
confusion matrix:
[[1004   10]
 [  58  202]]
              precision    recall  f1-score   support

           0     0.9454    0.9901    0.9672      1014
           1     0.9528    0.7769    0.8559       260

    accuracy                         0.9466      1274
   macro avg     0.9491    0.8835    0.9116      1274
weighted avg     0.9469    0.9466    0.9445      1274


[dual_stream_final_predictions_on_adv_test] acc=0.9396 f1=0.8344
confusion matrix:
[[1003   11]
 [  66  194]]
              precision    recall  f1-score   support

           0     0.9383    0.9892    0.9630      1014
           1     0.9463    0.7462    0.8344       260

    accuracy                         0.9396      1274
   macro avg     0.9423    0.8677    0.8987      1274
weighted avg     0.9399    0.9396    0.9368      1274


[dual_stream_final_predictions_on_combined_test] acc=0.9431 f1=0.8453
confusion matrix:
[[2007   21]
 [ 124  396]]
      

,model_eval,acc,f1
0,dual_stream_final_predictions_on_clean_test,0.946625,0.855932
1,dual_stream_final_predictions_on_adv_test,0.939560,0.834409
2,dual_stream_final_predictions_on_combined_test,0.943093,0.845251



Dual-stream attack-detection summary:


,attack,confidence_threshold,disagreement_threshold,FPR,TPR,F1,clean_model_acc_on_adv,guardian_model_acc_on_clean,guardian_model_acc_on_adv
0,FGM,0.2,0.55,0.032182,0.887755,0.924775,0.139717,0.946625,0.93956



Gate reason counts on clean test:


,reason,count
0,none,1233
1,disagreement,27
2,"disagreement,high_confidence_disagreement",14



Gate reason counts on adversarial test:


,reason,count
0,disagreement,994
1,none,143
2,"disagreement,high_confidence_disagreement",137


In [86]:

# Save dual-stream outputs
dual_stream_dir = Path(r"..\Results\NeuralNetworksResults")
dual_stream_dir.mkdir(parents=True, exist_ok=True)

dual_stream_prediction_summary_path = dual_stream_dir / "nn_dual_stream_prediction_summary.csv"
dual_stream_detection_path = dual_stream_dir / "nn_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(dual_stream_dir / "nn_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(dual_stream_dir / "nn_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(dual_stream_dir / "nn_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")


Saved: ..\Results\NeuralNetworksResults\nn_dual_stream_prediction_summary.csv
Saved: ..\Results\NeuralNetworksResults\nn_dual_stream_detection_results.csv
